# Brain2Simulator – Parameter Modulation Test

This notebook loads neuron and synapse parameters from `parameters.json`,
lets you tweak them in-cell, and runs a short Brian2 simulation so you can
inspect the membrane-potential trace and spike raster.

In [ ]:
import json

import matplotlib.pyplot as plt
from brian2 import (
    start_scope,
    Network,
    StateMonitor,
    SpikeMonitor,
    run,
    ms,
    defaultclock,
    seed,
)

from neuron_model import create_neuron_group, load_params
from synapse_model import create_synapses

## 1. Load base parameters

In [ ]:
# Load the default parameters and display them
params = load_params()
print(json.dumps(params, indent=2))

## 2. Modify parameters (optional)

Edit any value in the `params` dict below to explore the model behaviour.

In [ ]:
# Example: increase external current and lower the firing threshold
params["neuron"]["I_ext"] = 2.0       # nA
params["neuron"]["V_threshold"] = -52.0  # mV
params["simulation"]["duration"] = 200.0  # ms

## 3. Build and run the network

In [ ]:
start_scope()

sim_p = params["simulation"]
defaultclock.dt = sim_p["dt"] * ms
seed(sim_p["seed"])

N_input  = sim_p["N_input"]
N_output = sim_p["N_output"]
duration = sim_p["duration"] * ms

# Create neuron groups
input_layer  = create_neuron_group(N_input,  params=params)
output_layer = create_neuron_group(N_output, params=params)

# Create memristive synapses (input → output)
synapses = create_synapses(input_layer, output_layer, params=params)

# Monitors
spike_mon_in  = SpikeMonitor(input_layer)
spike_mon_out = SpikeMonitor(output_layer)
state_mon     = StateMonitor(output_layer, "V", record=[0, 1, 2])

net = Network(
    input_layer, output_layer, synapses,
    spike_mon_in, spike_mon_out, state_mon,
)
net.run(duration)

print(f"Input  spikes : {spike_mon_in.num_spikes}")
print(f"Output spikes : {spike_mon_out.num_spikes}")

## 4. Visualise results

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9))

# --- Spike raster: input layer ---
axes[0].plot(spike_mon_in.t / ms, spike_mon_in.i, "|k", markersize=2)
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Neuron index")
axes[0].set_title("Input layer spike raster")

# --- Spike raster: output layer ---
axes[1].plot(spike_mon_out.t / ms, spike_mon_out.i, "|r", markersize=4)
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Neuron index")
axes[1].set_title("Output layer spike raster")

# --- Membrane potential: first 3 output neurons ---
for idx in range(3):
    axes[2].plot(
        state_mon.t / ms,
        state_mon.V[idx] / 1e-3,   # convert V → mV
        label=f"neuron {idx}",
    )
axes[2].set_xlabel("Time (ms)")
axes[2].set_ylabel("Membrane potential (mV)")
axes[2].set_title("Output layer membrane potential")
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 5. Save modified parameters back to JSON

Uncomment the cell below to persist your changes to `parameters.json`.

In [ ]:
# with open("parameters.json", "w") as f:
#     json.dump(params, f, indent=4)
# print("parameters.json updated.")